<a href="https://colab.research.google.com/github/PhysShell/Easy-Wav2Lip/blob/v8.3/Easy_Wav2Lip_v8.3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Welcome to my Easy Wav2Lip colab!

My goal is to make lipsyncing with this tool easy, fast and great looking!

Please view the GitHub for instructions: [https://github.com/anothermartz/Easy-Wav2Lip](https://github.com/anothermartz/Easy-Wav2Lip?tab=readme-ov-file#best-practices)

In [9]:
!rm -rf installed.txt last_file.txt Easy-Wav2Lip

In [10]:
version = "v8.3"          # ← ветка Easy-Wav2Lip
# ------------------------------ STEP 1 ---------------------------------
#  One-click setup в Google Colab
# ----------------------------------------------------------------------

# 0) Проверяем, что сценарий не запускался ранее
import os, sys, time, warnings, torch, importlib.util, pathlib, textwrap, subprocess

if os.path.exists("installed.txt"):
    with open("../last_file.txt") as f:
        if f.readline().strip() == version:
            sys.exit(f"Easy-Wav2Lip {version} уже установлен в этом runtime")

# 1) Проверяем GPU
print("🔍  GPU:", "OK" if torch.cuda.is_available() else "NOT FOUND")
if not torch.cuda.is_available():
    sys.exit('В Runtime нет GPU — в меню «Runtime ▸ Change runtime type» выберите GPU')

# 2) (необязательно) подключаем Google Drive
try:
    from google.colab import drive
    print("🔗  Подключаем Google Drive ...")
    drive.mount("/content/drive")
except Exception:
    print("Google Drive пропущен — продолжим без него")

# 3) Клонируем репозиторий
giturl = "https://github.com/anothermartz/Easy-Wav2Lip.git"
print("📥  Скачиваем Easy-Wav2Lip …")
!git clone -q -b $version $giturl
%cd Easy-Wav2Lip
!mkdir -p face_alignment temp

# 4) Системные пакеты, нужные dlib и cmake
print("🔧  apt install build-essential cmake …")
!apt-get -qq update
!apt-get -qq install build-essential cmake

# 5) Ставим ядро PyTorch (есть cu121-wheels)
print("🧩  pip install torch/vision/audio …")
!pip install -q --extra-index-url https://download.pytorch.org/whl/cu121 \
             torch==2.1.0+cu121 torchvision==0.16.0+cu121 torchaudio==2.1.0+cu121

# 6) dlib — готовый wheel под Py 3.11
print("🧩  pip install dlib-bin …")
!pip install -q dlib-bin==19.24.6

# 7) Остальные зависимости, но без переустановки torch*
print("🧩  pip install project requirements …")

# --- ПАТЧ requirements.txt --------------------------------------------
# 1. Удаляем старую строку dlib==…  (нам хватает dlib-bin)
!sed -i '/^dlib==/d' requirements.txt

# 2. Гарантируем NumPy < 2 (заменяем или добавляем строку)
!grep -q '^numpy==' requirements.txt \
    && sed -i 's/^numpy==.*/numpy==1.26.4/' requirements.txt \
    || echo 'numpy==1.26.4' >> requirements.txt
# ----------------------------------------------------------------------

!pip install -q --no-cache-dir --force-reinstall "numpy<2"
!pip install -q stack_data executing asttokens pure_eval jedi
!pip install -q -r requirements.txt --no-deps

# 8) патчим basicsr/data/degradations.py (старый импорт)
print("🩹  Патчим basicsr → torchvision.transforms.functional …")
degr = pathlib.Path(importlib.util.find_spec("basicsr").submodule_search_locations[0]) / "data" / "degradations.py"
degr.write_text(degr.read_text().replace("functional_tensor", "functional"))

# 9) Запуск install.py самого проекта
print("🚀  Запускаем install.py …")
!python install.py

# 10) помечаем успешную установку
with open("installed.txt", "w"), open("../last_file.txt", "w") as f2:
    f2.write(version)

print("✅  Установка завершена!  Теперь переходите к Step 2.")


🔍  GPU: OK
🔗  Подключаем Google Drive ...
Google Drive пропущен — продолжим без него
📥  Скачиваем Easy-Wav2Lip …
/content/Easy-Wav2Lip/Easy-Wav2Lip/Easy-Wav2Lip/Easy-Wav2Lip/Easy-Wav2Lip/Easy-Wav2Lip
🔧  apt install build-essential cmake …
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
🧩  pip install torch/vision/audio …
🧩  pip install dlib-bin …
🧩  pip install project requirements …
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 268.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
scikit-image 0.25.2 requires scipy>=1.11.4, b

In [16]:
if not os.path.exists('installed.txt'):
  sys.exit('Step 1 has not been run in this instance! Please run step 1 each time you disconnect from a runtime.')
time
############################## user inputs #####################################
#@markdown <h1>Step 2: Select inputs:</h1>

# @markdown On destktop: <h1></h1>Click the folder icon ( 📁 ) at the left edge of colab, find your file, right click, copy path, paste it below:
#@markdown<br></br>
# @markdown On mobile: <h1></h1>Tap the hamburger button ( ☰ ) at the top left, click show file browser, find your file, long press on it, copy path, paste below:
video_file = "/content/video.mp4" #@param {type:"string"}
vocal_file = "/content/audio.mp3" #@param {type:"string"}

#@markdown > Keep vocal_file blank if your video already has the desired speech audio encoded into it.
#@markdown # Quality
quality = "Enhanced" # @param ["Fast", "Improved", "Enhanced"]
#@markdown * <b><u>Fast</u></b>: Wav2Lip <br>
#@markdown * <b><u>Improved</u></b>: Wav2Lip with a feathered mask around the mouth to remove the square around the face <br>
#@markdown * <b><u>Enhanced</u></b>: Wav2Lip + mask + GFPGAN upscaling done on the face
#preview_quality = False #@param {type:"boolean"} - coming soon!
output_height = "full resolution" #@param ["half resolution", "full resolution", "480"] {allow-input: true}
use_previous_tracking_data = True #@param {type:"boolean"}
#@markdown Speeds up processing of the same video used multiple times - it should delete the last tracking file automatically when the video is changed but if it's failing after the first video, untick this box.

#@markdown
#------------------------------*Step 3*----------------------------------------!
#@markdown <h1>👈 Step 3:  Click the little circle play button on this cell! </h1> (Or press ctrl + F10) - Then wait for processing to complete.
# scale padding with resolution
#@markdown <br>

#@markdown ---
#@markdown <br>

#@markdown # [Advanced tweaking](https://github.com/anothermartz/Easy-Wav2Lip/tree/v7#advanced-tweaking) (optional) </h1>Just ignore all of this if you are new, or click the blue titles for instructions.
wav2lip_version = "Wav2Lip" # @param ["Wav2Lip", "Wav2Lip_GAN"]
nosmooth = True #@param {type:"boolean"}
#@markdown ### [Padding:](https://github.com/anothermartz/Easy-Wav2Lip/tree/v7#padding)</h1> (Up, Down, Left, Right) <br>
U = 0 #@param {type:"slider", min:-100, max:100, step:1}
D = 10 #@param {type:"slider", min:-100, max:100, step:1}
L = 0 #@param {type:"slider", min:-100, max:100, step:1}
R = 0 #@param {type:"slider", min:-100, max:100, step:1}
#@markdown <br>

#@markdown ### [Mask:](https://github.com/anothermartz/Easy-Wav2Lip/tree/v7#other-options)
size = 1.5 #@param {type:"slider", min:1, max:6, step:0.1}
feathering = 1 #@param {type:"slider", min:0, max:3, step:1}
mouth_tracking = False #@param {type:"boolean"}
debug_mask = False #@param {type:"boolean"}


#@markdown # [Other options:](https://github.com/anothermartz/Easy-Wav2Lip/tree/v7#other-options)

batch_process = False #@param {type:"boolean"}
output_suffix = "_Easy-Wav2Lip" #@param {type:"string"}
include_settings_in_suffix = False #@param {type:"boolean"}
preview_input = False #@param {type:"boolean"}
preview_settings = False #@param {type:"boolean"}
#@markdown preview_settings processes only one frame so you can see how it looks without doing the whole video
frame_to_preview = 100 # @param {type:"integer"}


import configparser

# Create a ConfigParser object
config = configparser.ConfigParser()

# Put all your variables in a dictionary
options = {
    'video_file': video_file,
    'vocal_file': vocal_file,
    'quality': quality,
    'output_height': output_height,
    'wav2lip_version': wav2lip_version,
    'use_previous_tracking_data': use_previous_tracking_data,
    'nosmooth': nosmooth
}
padding = {
    'U': U,
    'D': D,
    'L': L,
    'R': R
}
mask = {
    'size': size,
    'feathering': feathering,
    'mouth_tracking': mouth_tracking,
    'debug_mask': debug_mask
}
other = {
    'batch_process': batch_process,
    'output_suffix': output_suffix,
    'include_settings_in_suffix': include_settings_in_suffix,
    'preview_input': preview_input,
    'preview_settings': preview_settings,
    'frame_to_preview': frame_to_preview
}


# Add the dictionary to the ConfigParser object
config['OPTIONS'] = options
config['PADDING'] = padding
config['MASK'] = mask
config['OTHER'] = other

# Write the data to an INI file
with open('config.ini', 'w') as f:
    config.write(f)

!python run.py

from easy_functions import show_video
from IPython.display import Image
if preview_settings:
  if os.path.isfile(os.path.join('temp','preview.jpg')):
    display(Image(os.path.join('temp','preview.jpg')))
else:
  if os.path.isfile(os.path.join('temp','output.mp4')):
    print(f"Loading video preview...")
    show_video(os.path.join('temp','output.mp4'))

Processing video.mp4 using audio.mp3 for audio
imports loaded!     
Converting audio to .wav
analysing audio...
894 frames to process
detecting face in every frame: 100%|█████████████████████████████| 894/894 [00:05<00:00, 166.74it/s]
mask size: 1.5, feathering: 1
Loading gfpgan
Starting...
Processing Wav2Lip: 100%|█████████████████████████████████████████| 952/952 [05:38<00:00,  2.82it/s]
converting to final video
video_audio successfully lip synced! It will be found here:
/content/video_audio_Easy-Wav2Lip.mp4
Execution time: 5m 59s
Loading video preview...
